# TNB Frame Animation

Edit `r_sym` in the cell below to define your parametric curve, then run all cells.

The animation will render and play inline.

In [ ]:
from manim import *
import sympy as s
import numpy as np
from sympy import symbols, sqrt, lambdify, sin, cos, tan, exp, log, pi

t = symbols('t')

In [ ]:
# ── Edit your curve here ──────────────────────────────────────────
r_sym = s.Matrix([2*cos(3*t), 2*sin(t), 2*cos(2*t)])

# t range for the animation
t_start = 0
t_end   = 2 * float(pi)
# ─────────────────────────────────────────────────────────────────

In [ ]:
%%manim -qm TNBFrame

class TNBFrame(ThreeDScene):
    def construct(self):
        def TNB_get(func):
            v_sym = func.diff(t)
            speed_sym = sqrt(v_sym.dot(v_sym))
            T_sym = v_sym / speed_sym
            N_sym = T_sym.diff(t) / sqrt(T_sym.diff(t).dot(T_sym.diff(t)))
            B_sym = T_sym.cross(N_sym)
            T = lambdify(t, T_sym, 'numpy')
            N = lambdify(t, N_sym, 'numpy')
            B = lambdify(t, B_sym, 'numpy')
            return T, N, B

        # ── auto-scale curve to fit scene ──
        r_temp = lambdify(t, r_sym, 'numpy')
        n_samp = 300
        ts_samp = np.linspace(t_start, t_end, n_samp)
        pts = np.array([np.squeeze(r_temp(ti)) for ti in ts_samp])
        max_extent = np.max(np.abs(pts - pts.mean(axis=0)))
        scene_scale = 3.5 / max(max_extent, 1e-6)
        vec_len = 0.6

        r_vis = lambda ti: np.squeeze(r_temp(ti)) * scene_scale

        # ── axes sized to scaled curve ──
        sc_pts = pts * scene_scale
        pad = 0.5

        def ax_range(vals):
            mn, mx = vals.min(), vals.max()
            span = max(mx - mn, 0.5)
            return [mn - pad, mx + pad, round(span / 5, 2)]

        axes = ThreeDAxes(
            x_range=ax_range(sc_pts[:, 0]),
            y_range=ax_range(sc_pts[:, 1]),
            z_range=ax_range(sc_pts[:, 2]),
            x_length=8, y_length=8, z_length=6,
        )

        self.move_camera(phi=PI/3, theta=PI/4)
        self.add(axes)
        self.wait()

        C   = ParametricFunction(r_vis, t_range=[t_start, t_end], color=RED)
        t0  = MathTex(r'\vec{r}(t)=', color=RED)
        T_l = MathTex(r'\vec{T}', color=GREEN)
        N_l = MathTex(r'\vec{N}', color=BLUE)
        B_l = MathTex(r'\vec{B}', color=PURPLE)
        TNB_l = VGroup(T_l, N_l, B_l)

        self.add_fixed_in_frame_mobjects(t0.shift(3*UP + 3*RIGHT).scale(0.75))
        self.play(Create(C), run_time=5)
        self.wait()

        tracker = ValueTracker(t_start)
        T_f, N_f, B_f = TNB_get(r_sym)
        T = lambda ti: np.squeeze(T_f(ti))
        N = lambda ti: np.squeeze(N_f(ti))
        B = lambda ti: np.squeeze(B_f(ti))

        T_p = Vector(T(t_start) * vec_len, color=GREEN).shift(r_vis(t_start))
        T_p.add_updater(lambda m: m.become(
            Vector(T(tracker.get_value()) * vec_len, color=GREEN).shift(r_vis(tracker.get_value()))))
        N_p = Vector(N(t_start) * vec_len, color=BLUE).shift(r_vis(t_start))
        N_p.add_updater(lambda m: m.become(
            Vector(N(tracker.get_value()) * vec_len, color=BLUE).shift(r_vis(tracker.get_value()))))
        B_p = Vector(B(t_start) * vec_len, color=PURPLE).shift(r_vis(t_start))
        B_p.add_updater(lambda m: m.become(
            Vector(B(tracker.get_value()) * vec_len, color=PURPLE).shift(r_vis(tracker.get_value()))))

        self.add(VGroup(T_p, N_p, B_p))
        self.add_fixed_in_frame_mobjects(TNB_l.arrange(DOWN).next_to(t0, 2*RIGHT))
        self.wait()
        self.play(tracker.animate.set_value(t_end), run_time=10, rate_func=linear)
        self.wait()